# Конвертация YOLOv8 модели в формат RKNN

## dynamic=False
> Определяет, будет ли размер входа динамическим. Значение: Почему важно: Для RKNN обязательно нужно dynamic=False, потому что NPU плохо работает с переменными размерами.
- False: вход фиксированного размера (например, [1, 3, 640, 640]).
- True: вход может быть переменным (например, [1, 3, H, W]).

## simplify=True
> Упрощает ONNX-граф (удаляет лишние операции, объединяет блоки). Всегда ставить True, особенно перед импортом в RKNN Toolkit.
- Повышает производительность.
- Снижает размер файла.
- Устраняет потенциальные ошибки при конвертации.

## opset=12
> Указывает версию ONNX-операторов (opset). Некоторые версии RKNN Toolkit поддерживают только определённые версии opset. Версия 12 — стабильная и хорошо поддерживается большинством инструментов, включая RKNN Toolkit 1.x и 2.x.

In [1]:
from ultralytics import YOLO

In [2]:
model = YOLO('../Weight/yolo8s_300_split_aug_scale.pt')
model.export(format="onnx", dynamic=False, simplify=True, opset=11, imgsz=640)
# nms=False

Ultralytics 8.3.138 🚀 Python-3.10.12 torch-2.2.0 CPU (Cortex-A55)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '../Weight/yolo8s_300_split_aug_scale.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (21.5 MB)

ONNX: starting export with onnx 1.16.1 opset 11...
ONNX: slimming with onnxslim 0.1.53...
ONNX: export success ✅ 4.6s, saved as '../Weight/yolo8s_300_split_aug_scale.onnx' (42.7 MB)

Export complete (8.8s)
Results saved to /home/orangepi/Desktop/DetectPeople/Weight
Predict:         yolo predict task=detect model=../Weight/yolo8s_300_split_aug_scale.onnx imgsz=640  
Validate:        yolo val task=detect model=../Weight/yolo8s_300_split_aug_scale.onnx imgsz=640 data=./DataSet/SplitData/dataset.yaml  
Visualize:       https://netron.app


'../Weight/yolo8s_300_split_aug_scale.onnx'

# Скрипт для конвертации ONNX → RKNN

In [3]:
from rknn.api import RKNN

ONNX_MODEL = '../Weight/yolo8s_300_split_aug_scale.onnx'
RKNN_MODEL = '../Weight/yolo8s__3.rknn'
RKNN_MODEL_TXT = '/home/orangepi/Desktop/DetectPeople/Data/dataSet/dataset.txt'

rknn = RKNN()

I rknn-toolkit2 version: 2.3.2


In [4]:
print('Конфигурирование модели...')
ret = rknn.config(
    mean_values=[[0, 0, 0]], 
    std_values=[[255, 255, 255]], 
    target_platform='rk3588'  # укажи тут свою платформу, например rk3566, rk3568 и т.д.
)
if ret != 0:
    print('Ошибка конфигурации модели')
    exit(ret)

Конфигурирование модели...


In [5]:
print('Загрузка ONNX модели...')
ret = rknn.load_onnx(model=ONNX_MODEL)
if ret != 0:
    print('Ошибка загрузки ONNX модели')
    exit(ret)

Загрузка ONNX модели...


I Loading : 100%|██████████████████████████████████████████████| 140/140 [00:00<00:00, 11761.69it/s]


In [6]:
print('Компиляция модели...')
ret = rknn.build(do_quantization=False)
if ret != 0:
    print('Ошибка компиляции модели')
    exit(ret)

Компиляция модели...


I OpFusing 1 :  95%|███████████████████████████████████████████▋  | 95/100 [00:00<00:00, 171.16it/s]

I OpFusing 0 :  98%|█████████████████████████████████████████████ | 98/100 [00:00<00:00, 124.36it/s]

I OpFusing 0 :   0%|                                                        | 0/100 [00:01<?, ?it/s]

I OpFusing 2 : 100%|██████████████████████████████████████████████| 100/100 [00:01<00:00, 64.30it/s]

I OpFusing 2 : 100%|██████████████████████████████████████████████| 100/100 [00:03<00:00, 28.67it/s]
I rknn building ...
I rknn building done.


In [7]:
print('Экспорт RKNN модели...')
ret = rknn.export_rknn(RKNN_MODEL)
if ret != 0:
    print('Ошибка экспорта RKNN модели')
    exit(ret)

print('Готово!')

Экспорт RKNN модели...
Готово!
